# 面试问题：Agent 怎样通过 JIT Credential Broker 无密调用工具，并验证短期能力凭证？

        ## 可直接复述的回答主线

        1. Agent 运行时不应持有长期 API Key，而应向 Broker 申请与一次工具调用绑定的短期能力凭证。
2. Broker 先校验 tenant、agent、audience、scope 和审批，再签发带 sub、exp、jti 的最小权限 Token。
3. 工具端必须重新计算并常量时间比较签名，同时检查过期、受众、租户、主体、scope 和重放，不能只看 Token 是否存在。
4. 有副作用的 write scope 应要求审批票据，未审批或超出策略的申请在签发前拒绝。
5. 审计日志只保存 jti、凭证指纹和决策原因，不记录 Broker 密钥或完整 Token。
6. 生产应使用云工作负载身份、KMS/HSM 签名、mTLS、密钥轮换、撤销、时钟偏差与细粒度资源条件。

        后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例包含两个租户、两个 Agent 和六次工具调用，覆盖文档读取、CRM 写入、日历读取、越权薪资读取和缺少审批。Token 使用教学 HMAC 密钥真实签名与验证；密钥是假数据，只用于展示协议，不能当作生产密钥系统。

In [1]:
import base64  # 对签名载荷执行 URL-safe 编码。
import hashlib  # 生成 Token 指纹和确定性 jti。
import hmac  # 使用标准库执行真实 HMAC-SHA256 签名验证。
import json  # 对凭证 claims 做确定性规范序列化。
broker_secret = b"teaching-only-broker-secret-56"  # 定义仅供离线教学的假 HMAC 密钥且不会写入 Agent 状态。
policies = {("tenant-a", "agent-support"): {"docs-api": {"docs.read"}, "crm-api": {"crm.ticket.write"}}, ("tenant-b", "agent-sales"): {"calendar-api": {"calendar.read"}, "crm-api": {"crm.lead.write"}}}  # 定义租户、Agent、受众到允许 scope 的最小权限策略。
requests = [{"id": "cred-01", "tenant": "tenant-a", "agent": "agent-support", "audience": "docs-api", "scope": "docs.read", "resource": "kb/refund", "approved": True, "now": 1000, "expected": True}, {"id": "cred-02", "tenant": "tenant-a", "agent": "agent-support", "audience": "crm-api", "scope": "crm.ticket.write", "resource": "ticket/T-7", "approved": True, "now": 1002, "expected": True}, {"id": "cred-03", "tenant": "tenant-a", "agent": "agent-support", "audience": "payroll-api", "scope": "payroll.read", "resource": "salary/all", "approved": True, "now": 1004, "expected": False}, {"id": "cred-04", "tenant": "tenant-b", "agent": "agent-sales", "audience": "calendar-api", "scope": "calendar.read", "resource": "calendar/team", "approved": True, "now": 1006, "expected": True}, {"id": "cred-05", "tenant": "tenant-b", "agent": "agent-sales", "audience": "crm-api", "scope": "crm.lead.write", "resource": "lead/L-9", "approved": False, "now": 1008, "expected": False}, {"id": "cred-06", "tenant": "tenant-b", "agent": "agent-sales", "audience": "crm-api", "scope": "crm.lead.write", "resource": "lead/L-10", "approved": True, "now": 1010, "expected": True}]  # 定义六次具有权限、审批和资源语义的工具调用。
print("教学实验输入：JIT 工具凭证申请")  # 标记下方为脱敏离线授权案例。
print("请求      tenant    agent          audience      scope              approved  expected")  # 输出申请预览表头。
for request in requests:  # 逐条展示授权决策所需字段。
    print(f"{request['id']:<9} {request['tenant']:<9} {request['agent']:<14} {request['audience']:<13} {request['scope']:<18} {str(request['approved']):>8} {str(request['expected']):>9}")  # 输出当前工具调用的权限语义。
agent_runtime_state = {"conversation_id": "conv-56", "long_lived_secrets": [], "broker_secret": None}  # 明确 Agent 进程不保存长期凭证或 Broker 密钥。
print("Agent运行时秘密状态：", agent_runtime_state)  # 展示 secretless 边界而不泄露完整 Token。

教学实验输入：JIT 工具凭证申请
请求      tenant    agent          audience      scope              approved  expected
cred-01   tenant-a  agent-support  docs-api      docs.read              True      True
cred-02   tenant-a  agent-support  crm-api       crm.ticket.write       True      True
cred-03   tenant-a  agent-support  payroll-api   payroll.read           True     False
cred-04   tenant-b  agent-sales    calendar-api  calendar.read          True      True
cred-05   tenant-b  agent-sales    crm-api       crm.lead.write        False     False
cred-06   tenant-b  agent-sales    crm-api       crm.lead.write         True      True
Agent运行时秘密状态： {'conversation_id': 'conv-56', 'long_lived_secrets': [], 'broker_secret': None}


## 2. Baseline / 基线：共享一个长期全权限 Token

基线只检查静态 Token 字符串存在就执行工具。六次调用都会放行，因此越权薪资读取和未审批 CRM 写入也被执行。

In [2]:
static_credential = {"token_present": True, "audience": "*", "scopes": {"*"}, "expires_at": 999999}  # 模拟被所有 Agent 共享的长期全权限凭证。
baseline_rows = []  # 保存六次调用的朴素授权结果。
for request in requests:  # 对同一批工具申请执行只看存在性的基线。
    executed = bool(static_credential["token_present"])  # 错误地把凭证存在等价为允许执行。
    baseline_rows.append({"id": request["id"], "executed": executed, "correct": executed == request["expected"], "reason": "static_token_present"})  # 保存执行和期望对照。
baseline_accuracy = sum(row["correct"] for row in baseline_rows) / len(baseline_rows)  # 计算基线授权决策准确率。
print("Baseline 长期凭证授权")  # 标记下表没有签名、受众或 scope 验证。
print("请求      executed  expected  correct  reason")  # 输出基线结果表头。
for row, request in zip(baseline_rows, requests):  # 逐调用展示越权放行。
    print(f"{row['id']:<9} {str(row['executed']):>8} {str(request['expected']):>9} {str(row['correct']):>8}  {row['reason']}")  # 输出当前基线决策。

Baseline 长期凭证授权
请求      executed  expected  correct  reason
cred-01       True      True     True  static_token_present
cred-02       True      True     True  static_token_present
cred-03       True     False    False  static_token_present
cred-04       True      True     True  static_token_present
cred-05       True     False    False  static_token_present
cred-06       True      True     True  static_token_present


## 3. 底层实现：策略门禁、JIT 签发与工具端完整验证

不调用认证框架。Broker 生成规范 JSON、HMAC-SHA256 和 60 秒 claims；工具重新计算签名，再逐项核对 exp、aud、sub、tenant、scope 和 jti。

In [3]:
def encode_payload(payload):  # 把 claims 规范序列化为 URL-safe 字符串。
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")  # 固定字段顺序和编码避免签名歧义。
    return base64.urlsafe_b64encode(canonical).decode("ascii").rstrip("=")  # 生成不含填充的 URL-safe 载荷。
def decode_payload(encoded):  # 从 URL-safe 字符串还原 claims。
    padding = "=" * (-len(encoded) % 4)  # 补回 Base64 所需填充。
    return json.loads(base64.urlsafe_b64decode(encoded + padding).decode("utf-8"))  # 解码并解析 JSON claims。
def sign_encoded(encoded):  # 用 Broker 假密钥计算真实 HMAC-SHA256。
    return hmac.new(broker_secret, encoded.encode("ascii"), hashlib.sha256).hexdigest()  # 返回六十四位十六进制消息认证码。
def issue_credential(request, ttl_seconds=60):  # 根据策略和审批签发一次短期能力凭证。
    allowed_scopes = policies.get((request["tenant"], request["agent"]), {}).get(request["audience"], set())  # 查找主体在目标工具上的最小权限集合。
    if request["scope"] not in allowed_scopes:  # 检查申请 scope 是否越过租户策略。
        return None, "policy_denied", None  # 在签名前拒绝越权申请。
    if request["scope"].endswith(".write") and not request["approved"]:  # 对有副作用写操作检查审批票据。
        return None, "approval_required", None  # 未审批写操作不签发 Token。
    jti_source = f"{request['id']}|{request['tenant']}|{request['agent']}|{request['now']}"  # 构造确定性教学 jti 输入。
    payload = {"iss": "jit-broker", "sub": request["agent"], "tenant": request["tenant"], "aud": request["audience"], "scope": request["scope"], "resource": request["resource"], "iat": request["now"], "exp": request["now"] + ttl_seconds, "jti": hashlib.sha256(jti_source.encode("utf-8")).hexdigest()[:16]}  # 生成最小权限且短期的 claims。
    encoded = encode_payload(payload)  # 规范编码待签名载荷。
    signature = sign_encoded(encoded)  # 对完整载荷计算真实消息认证码。
    token = f"{encoded}.{signature}"  # 拼接载荷和签名形成教学 Token。
    fingerprint = hashlib.sha256(token.encode("ascii")).hexdigest()[:12]  # 生成可审计但不可直接使用的 Token 指纹。
    return token, "issued", {"jti": payload["jti"], "fingerprint": fingerprint, "exp": payload["exp"]}  # 返回短期 Token 和脱敏审计字段。
def verify_credential(token, expected, now, seen_jtis=None):  # 在工具端完整验证签名和全部调用绑定 claims。
    parts = token.split(".")  # 分离编码载荷和 HMAC。
    if len(parts) != 2:  # 拒绝结构不完整的 Token。
        return False, "malformed", None  # 返回格式错误且不解析 claims。
    encoded, supplied_signature = parts  # 读取签名覆盖的载荷和客户端提供的 HMAC。
    expected_signature = sign_encoded(encoded)  # 使用工具端受信密钥重新计算 HMAC。
    if not hmac.compare_digest(supplied_signature, expected_signature):  # 常量时间比较防止只看签名存在。
        return False, "bad_signature", None  # 拒绝任何载荷或签名篡改。
    payload = decode_payload(encoded)  # 仅在签名有效后解析受信 claims。
    checks = [(payload.get("iss") == "jit-broker", "bad_issuer"), (now < payload.get("exp", 0), "expired"), (payload.get("aud") == expected["audience"], "audience_mismatch"), (payload.get("sub") == expected["agent"], "subject_mismatch"), (payload.get("tenant") == expected["tenant"], "tenant_mismatch"), (payload.get("scope") == expected["scope"], "scope_mismatch"), (payload.get("resource") == expected["resource"], "resource_mismatch")]  # 定义工具端必须逐项满足的上下文绑定。
    for passed, reason in checks:  # 依次验证发行方、时效、受众、主体和能力范围。
        if not passed:  # 检查当前 claim 是否不匹配。
            return False, reason, payload  # 返回明确拒绝原因供审计。
    if seen_jtis is not None and payload["jti"] in seen_jtis:  # 检查一次性 jti 是否已经被消费。
        return False, "replay", payload  # 阻止同一短期能力凭证再次使用。
    if seen_jtis is not None:  # 只在调用方启用重放防护时记录 jti。
        seen_jtis.add(payload["jti"])  # 原子消费当前教学 jti。
    return True, "verified", payload  # 返回签名和所有 claims 均通过的结果。
first_token, first_issue_reason, first_audit = issue_credential(requests[0])  # 为文档读取申请生成一枚示例凭证。
first_valid, first_verify_reason, first_claims = verify_credential(first_token, requests[0], requests[0]["now"], set())  # 在工具端真实验证示例 Token。
print("cred-01签发审计=", first_audit)  # 只输出 jti、过期时间和指纹而不打印完整 Token。
print("cred-01已验证claims=", first_claims, "decision=", first_valid, first_verify_reason)  # 展示签名之后逐项验证的受信字段。

cred-01签发审计= {'jti': '3a43f744efb455d0', 'fingerprint': 'e7323878ffa5', 'exp': 1060}
cred-01已验证claims= {'aud': 'docs-api', 'exp': 1060, 'iat': 1000, 'iss': 'jit-broker', 'jti': '3a43f744efb455d0', 'resource': 'kb/refund', 'scope': 'docs.read', 'sub': 'agent-support', 'tenant': 'tenant-a'} decision= True verified


## 4. 逐请求结果与结果解读

每条允许调用获得不同 jti，工具只在完整验证后执行；越权和缺审批请求根本不会拿到 Token。结果表只展示指纹，不泄露完整凭证。

In [4]:
consumed_jtis = set()  # 模拟工具端共享的一次性 jti 存储。
corrected_rows = []  # 保存六次申请的签发和执行结果。
issued_tokens = {}  # 仅在当前内存中保存允许请求的短期 Token 供失败实验。
for request in requests:  # 逐条执行 Broker 策略和工具验证。
    token, issue_reason, audit = issue_credential(request)  # 先尝试签发最小权限凭证。
    if token is None:  # 检查策略或审批是否阻止签发。
        executed = False  # 无凭证时工具不会被调用。
        verify_reason = "not_issued"  # 标记没有进入工具验证路径。
        fingerprint = "-"  # 未签发请求没有 Token 指纹。
    else:  # 对成功签发的 Token 执行工具端验证。
        issued_tokens[request["id"]] = token  # 保存短期 Token 供后续篡改和过期实验。
        executed, verify_reason, payload = verify_credential(token, request, request["now"], consumed_jtis)  # 验证签名、上下文和重放。
        fingerprint = audit["fingerprint"]  # 读取脱敏 Token 指纹。
    corrected_rows.append({"id": request["id"], "issued": token is not None, "issue_reason": issue_reason, "verify_reason": verify_reason, "executed": executed, "correct": executed == request["expected"], "fingerprint": fingerprint})  # 保存逐请求授权链路。
corrected_accuracy = sum(row["correct"] for row in corrected_rows) / len(corrected_rows)  # 计算 JIT 授权决策准确率。
print("请求      issued  issue_reason       verify_reason      executed expected correct fingerprint")  # 输出同数据逐请求对照表头。
for row, request in zip(corrected_rows, requests):  # 逐条展示签发、验证和最终执行。
    print(f"{row['id']:<9} {str(row['issued']):>6}  {row['issue_reason']:<18} {row['verify_reason']:<18} {str(row['executed']):>8} {str(request['expected']):>8} {str(row['correct']):>7} {row['fingerprint']}")  # 输出当前请求的完整决策链。
print(f"结果解读：静态长期凭证决策准确率={baseline_accuracy:.1%}，JIT签发加工具验证={corrected_accuracy:.1%}；拒绝发生在敏感工具执行之前。")  # 解释最小权限和审批门禁的效果。

请求      issued  issue_reason       verify_reason      executed expected correct fingerprint
cred-01     True  issued             verified               True     True    True e7323878ffa5
cred-02     True  issued             verified               True     True    True 4fa6683b8f8b
cred-03    False  policy_denied      not_issued            False    False    True -
cred-04     True  issued             verified               True     True    True b9cc45233317
cred-05    False  approval_required  not_issued            False    False    True -
cred-06     True  issued             verified               True     True    True a5d587bb8d99
结果解读：静态长期凭证决策准确率=66.7%，JIT签发加工具验证=100.0%；拒绝发生在敏感工具执行之前。


## 5. 失败案例与修正：有签名字段不等于签名有效

修改已签发 Token 的 audience 后保留原 HMAC，字符串仍有“签名字段”，但重新计算会得到 `bad_signature`。另外展示错误 audience、过期和 jti 重放三种独立拒绝。

In [5]:
original_token = issued_tokens["cred-01"]  # 读取一枚已正常签发的文档读取 Token。
original_encoded, original_signature = original_token.split(".")  # 分离原载荷和真实 HMAC。
tampered_payload = decode_payload(original_encoded)  # 解码载荷以模拟攻击者修改字段。
tampered_payload["aud"] = "payroll-api"  # 把允许的 docs-api 受众篡改为高敏 payroll-api。
tampered_encoded = encode_payload(tampered_payload)  # 重新编码被修改的 claims。
tampered_token = f"{tampered_encoded}.{original_signature}"  # 保留旧签名以形成“签名存在但无效”的 Token。
tampered_valid, tampered_reason, tampered_claims = verify_credential(tampered_token, requests[0], requests[0]["now"], set())  # 真实重新计算 HMAC 并拒绝篡改。
wrong_audience_expected = {**requests[0], "audience": "crm-api"}  # 构造工具收到正确签名但受众不匹配的调用上下文。
wrong_audience_valid, wrong_audience_reason, wrong_audience_claims = verify_credential(original_token, wrong_audience_expected, requests[0]["now"], set())  # 验证 audience 绑定而非只验签名。
expired_valid, expired_reason, expired_claims = verify_credential(original_token, requests[0], first_claims["exp"] + 1, set())  # 在过期时间后验证同一正确签名 Token。
replay_store = set()  # 创建隔离的 jti 消费集合演示重放。
first_use_valid, first_use_reason, first_use_claims = verify_credential(original_token, requests[0], requests[0]["now"], replay_store)  # 第一次消费 jti 应成功。
replay_valid, replay_reason, replay_claims = verify_credential(original_token, requests[0], requests[0]["now"], replay_store)  # 第二次使用同一 jti 应被拒绝。
print(f"错误行为：篡改后仍有signature字段={bool(original_signature)}；如果只看存在性会放行。")  # 明确复现危险的浅层检查。
print(f"修正行为：tampered={tampered_valid}/{tampered_reason}，wrong_audience={wrong_audience_valid}/{wrong_audience_reason}，expired={expired_valid}/{expired_reason}，replay={replay_valid}/{replay_reason}")  # 展示真实验签和 claims 验证的四种拒绝。

错误行为：篡改后仍有signature字段=True；如果只看存在性会放行。
修正行为：tampered=False/bad_signature，wrong_audience=False/audience_mismatch，expired=False/expired，replay=False/replay


## 6. 生产边界

教学 HMAC 让 Broker 与工具共享密钥，不适合大规模信任域。生产应优先使用工作负载身份、短期非对称签名、JWKS/KMS/HSM、mTLS、密钥轮换和撤销；jti 消费需要原子存储，还要处理时钟偏差、资源级条件、审计脱敏与 Broker 高可用。

In [6]:
credential_diagnostics = {"requests": len(requests), "issued": sum(row["issued"] for row in corrected_rows), "denied_before_tool": sum(not row["issued"] for row in corrected_rows), "verified_executions": sum(row["executed"] for row in corrected_rows), "decision_accuracy": corrected_accuracy, "token_bodies_logged": 0, "agent_long_lived_secrets": len(agent_runtime_state["long_lived_secrets"])}  # 汇总授权、泄密和审计指标。
print("生产监控快照：", credential_diagnostics)  # 输出 JIT Credential Broker 应持续观察的指标。

生产监控快照： {'requests': 6, 'issued': 4, 'denied_before_tool': 2, 'verified_executions': 4, 'decision_accuracy': 1.0, 'token_bodies_logged': 0, 'agent_long_lived_secrets': 0}


## 7. 最小回归测试

断言覆盖输入规模、决策改善、真实验签、最小权限、过期与重放。

In [7]:
assert len(requests) >= 5 and len(policies) >= 2  # 保证案例包含足够调用和多个租户策略。
assert corrected_accuracy > baseline_accuracy and corrected_accuracy == 1.0  # 保证同数据 JIT 决策优于长期全权限基线。
assert first_valid and hmac.compare_digest(original_signature, sign_encoded(original_encoded))  # 保证正常 Token 的 HMAC 是真实重新计算后通过。
assert not tampered_valid and tampered_reason == "bad_signature"  # 保证签名字段存在不能绕过载荷篡改检测。
assert not wrong_audience_valid and wrong_audience_reason == "audience_mismatch" and not expired_valid and expired_reason == "expired"  # 保证受众和有效期都在工具端绑定验证。
assert first_use_valid and not replay_valid and replay_reason == "replay"  # 保证一次性 jti 的第二次使用被阻止。
assert agent_runtime_state["broker_secret"] is None and not agent_runtime_state["long_lived_secrets"]  # 保证 Agent 运行时没有长期密钥。